# MLOps - Caso de uso 1: Wine (CSV independiente)

**Pregunta de negocio:** ¿se puede identificar el cultivar de un vino a partir de su análisis químico?

Clasificación multiclase (3 clases) sobre 178 muestras y 13 variables químicas. Los datos se leen de un CSV (`wine.csv`, ruta configurable con la variable de entorno `DATA_PATH`). El CSV debe tener las 13 variables, la columna `target` y, opcionalmente, una columna `id` (se excluye de las features).

Notebook compatible con la plataforma MLOps (Papermill + MLflow). Tags usados:
`parameters`, `mlops:config`, `mlops:data`, `mlops:preprocessing`, `mlops:training`, `mlops:evaluation`, `mlops:export`.

## Parámetros
Papermill sobrescribe esta celda en tiempo de ejecución.

In [ ]:
# Papermill injected parameters (do not edit this cell manually)
# These are overwritten at runtime by the pipeline.
MODEL_OUTPUT_PATH = "./model.joblib"
PIPELINE_ID = "local-dev"
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_RUN_ID = "local-dev"

## Configuración
Nombre y versión con los que la plataforma registra el modelo.

In [ ]:
# ============================================================
# mlops:config - Metadatos del modelo
# ============================================================
MODEL_NAME = "wine-classifier"
VERSION = "1"

print(f"Modelo: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")

## Datos y análisis exploratorio
Se carga el dataset y se revisan clases, rangos y correlaciones para justificar el modelado. Si la plataforma define `GOLD_DB_PATH` y `GOLD_TABLE`, se lee la capa gold (DuckDB, solo lectura).

In [ ]:
# ============================================================
# mlops:data - Carga de datos desde un CSV independiente
# ============================================================
import os
import numpy as np
import pandas as pd

TARGET_COL = "target"
df = None

# Ruta del CSV: variable de entorno DATA_PATH (por defecto ./wine.csv, ruta relativa).
DATA_PATH = os.environ.get("DATA_PATH", "wine.csv")
FUENTE_DATOS = f"CSV: {DATA_PATH}"

# Si la plataforma expone la capa gold, tiene prioridad sobre el CSV local.
GOLD_DB_PATH = os.environ.get("GOLD_DB_PATH", "")
GOLD_TABLE = os.environ.get("GOLD_TABLE", "")
if GOLD_DB_PATH and GOLD_TABLE:
    try:
        # TODO: ajustar a la convención real de lectura de gold de la plataforma
        import duckdb
        con = duckdb.connect(GOLD_DB_PATH, read_only=True)
        try:
            df = con.execute(f'SELECT * FROM "{GOLD_TABLE}"').df()
        finally:
            con.close()
        FUENTE_DATOS = f"gold DuckDB: {GOLD_DB_PATH} :: {GOLD_TABLE}"
    except Exception as exc:
        print(f"No se pudo leer gold ({exc}); se usa el CSV.")
        df = None

if df is None:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(
            f"No se encontró el CSV en '{DATA_PATH}'. Coloca wine.csv junto al notebook "
            "(en Colab: sube el archivo desde el panel de archivos de la izquierda) "
            "o define la variable de entorno DATA_PATH."
        )
    df = pd.read_csv(DATA_PATH)

# Se excluye cualquier columna marca de agua / identificador (no es predictiva)
for _col in ("id", "ID", "Id"):
    if _col in df.columns:
        df = df.drop(columns=[_col])
        print(f"Columna '{_col}' excluida de las features (marca de agua).")

if TARGET_COL not in df.columns:
    raise KeyError(f"No se encontró la columna objetivo '{TARGET_COL}'. Columnas: {list(df.columns)}")

X = df.drop(columns=[TARGET_COL]).astype(float)
y = df[TARGET_COL].astype(int)
FEATURE_NAMES = list(X.columns)

# ------------------------------------------------------------
# Análisis exploratorio breve (solo prints, sin gráficos)
# ------------------------------------------------------------
print(f"Fuente de datos: {FUENTE_DATOS}")
print(f"Dimensiones: {X.shape[0]} filas x {X.shape[1]} variables")
print(f"Nulos totales: {int(X.isna().sum().sum())}")

print("\nDistribución de clases (conteo y %):")
dist = y.value_counts().sort_index()
print(pd.DataFrame({"conteo": dist, "porcentaje": (dist / len(y) * 100).round(1)}))

print("\nRangos de las variables (min / max / media / desv.):")
print(X.describe().loc[["min", "max", "mean", "std"]].T.round(2))

print("\nTop 5 variables por correlación absoluta con el target:")
corr_target = X.corrwith(y).abs().sort_values(ascending=False)
print(corr_target.head(5).round(3))

print("\nPares de variables más correlacionadas entre sí (|r| > 0.7):")
corr = X.corr().abs()
pares = [(a, b, round(float(corr.loc[a, b]), 3))
         for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]
         if corr.loc[a, b] > 0.7]
print(pares if pares else "Ninguno")

print("\nConclusión: clases moderadamente balanceadas y escalas muy distintas entre "
      "variables (p. ej. proline vs. hue), por lo que se estandariza y se usa "
      "validación estratificada.")

## Preprocesamiento
División estratificada 80/20 y estandarización. El escalador se ajusta solo con train (sin *data leakage*).

In [ ]:
# ============================================================
# mlops:preprocessing - División train/test y preprocesamiento
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# El escalador se define aquí pero se ajusta SOLO con train, dentro de cada Pipeline
# (durante la CV se reajusta por fold, evitando data leakage).
preprocessor = ColumnTransformer(
    transformers=[("scaler", StandardScaler(), FEATURE_NAMES)],
    remainder="drop",
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Distribución en train:", y_train.value_counts(normalize=True).sort_index().round(3).to_dict())
print("Distribución en test: ", y_test.value_counts(normalize=True).sort_index().round(3).to_dict())

## Entrenamiento
Se comparan tres candidatos con validación cruzada estratificada de 5 folds y se elige el de mayor accuracy media.

In [ ]:
# ============================================================
# mlops:training - Entrenamiento y selección de modelo
# ============================================================
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

candidatos = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "SVC": SVC(kernel="rbf", C=1.0, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas = []
for nombre, modelo in candidatos.items():
    pipe = Pipeline([("prep", clone(preprocessor)), ("model", modelo)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    filas.append({"modelo": nombre, "cv_accuracy_mean": scores.mean(), "cv_accuracy_std": scores.std()})

tabla_cv = pd.DataFrame(filas).sort_values("cv_accuracy_mean", ascending=False).reset_index(drop=True)
print("Comparativa de modelos (CV estratificada, 5 folds, sobre train):")
print(tabla_cv.round(4).to_string(index=False))

best_name = tabla_cv.loc[0, "modelo"]
cv_accuracy_mean = float(tabla_cv.loc[0, "cv_accuracy_mean"])
model = Pipeline([("prep", clone(preprocessor)), ("model", clone(candidatos[best_name]))])
model.fit(X_train, y_train)
print(f"\nMejor modelo: {best_name} (cv_accuracy_mean={cv_accuracy_mean:.4f})")
print("Entrenamiento completo.")

## Evaluación
Métricas sobre el conjunto de test y registro en MLflow. La métrica `accuracy` es obligatoria: la plataforma la usa para el despliegue automático.

In [ ]:
# ============================================================
# mlops:evaluation - Métricas y registro en MLflow
# ============================================================
# La plataforma lee la métrica 'accuracy' para decidir el despliegue automático.
import sys
import socket
import subprocess
from urllib.parse import urlparse

import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# MLFLOW_RUN_ID == "local-dev" significa ejecución manual (fuera de la plataforma).
# Solo en ese modo el notebook se adapta al entorno; bajo el pipeline real
# cualquier fallo de MLflow se propaga para no ocultar errores de registro.
LOCAL_DEV = MLFLOW_RUN_ID == "local-dev"

try:
    import mlflow
except ImportError:
    if not LOCAL_DEV:
        raise
    print("mlflow no está instalado; se intenta instalar (solo en modo local)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlflow"])
        import mlflow
    except Exception as exc:
        raise ImportError("No se pudo instalar mlflow automáticamente. Ejecuta: pip install mlflow") from exc


def _servidor_accesible(uri, timeout=3):
    # Solo comprueba URIs http(s); otros esquemas se consideran accesibles.
    if not uri.startswith(("http://", "https://")):
        return True
    p = urlparse(uri)
    puerto = p.port or (443 if p.scheme == "https" else 80)
    try:
        socket.create_connection((p.hostname, puerto), timeout=timeout).close()
        return True
    except OSError:
        return False


tracking_uri = MLFLOW_TRACKING_URI
if LOCAL_DEV and (tracking_uri.startswith("file:") or not _servidor_accesible(tracking_uri)):
    # El backend de archivos está en mantenimiento en MLflow reciente y localhost:5000
    # no existe fuera de la plataforma: se usa SQLite local para la prueba.
    print(f"[modo local] '{MLFLOW_TRACKING_URI}' no es utilizable; se usa sqlite:///mlflow.db")
    tracking_uri = "sqlite:///mlflow.db"

mlflow.set_tracking_uri(tracking_uri)
print(f"MLflow tracking URI en uso: {tracking_uri}")

y_pred = model.predict(X_test)
score = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(classification_report(y_test, y_pred, zero_division=0))
print("Matriz de confusión (filas=real, columnas=predicho):")
print(confusion_matrix(y_test, y_pred))
print(f"Accuracy: {score:.4f} | F1 macro: {f1_macro:.4f}")

# Hiperparámetros escalares del modelo elegido
hiperparametros = {
    k: v for k, v in model.named_steps["model"].get_params().items()
    if isinstance(v, (int, float, str, bool)) or v is None
}

# Usa el run iniciado por el pipeline si existe; si no, inicia uno nuevo
run_ctx = mlflow.start_run(run_id=MLFLOW_RUN_ID) if MLFLOW_RUN_ID != "local-dev" else mlflow.start_run()
with run_ctx:
    mlflow.log_metric("accuracy", score)
    mlflow.log_metric("f1_macro", f1_macro)
    mlflow.log_metric("precision_macro", precision_macro)
    mlflow.log_metric("recall_macro", recall_macro)
    mlflow.log_metric("cv_accuracy_mean", cv_accuracy_mean)
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("model_version", VERSION)
    mlflow.log_param("best_estimator", best_name)
    mlflow.log_param("data_source", FUENTE_DATOS[:250])
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    for k, v in hiperparametros.items():
        mlflow.log_param(f"hp_{k}", v)
print("Métricas registradas en MLflow.")

## Exportación
Se guarda el Pipeline completo en `MODEL_OUTPUT_PATH` y se verifica recargándolo.

In [ ]:
# ============================================================
# mlops:export - Guardar el artefacto del modelo
# ============================================================
# Se exporta el Pipeline COMPLETO (escalado + modelo): el servidor de inferencia
# recibe variables crudas, en el orden indicado abajo.
import os
import joblib

_dir = os.path.dirname(MODEL_OUTPUT_PATH)
if _dir:
    os.makedirs(_dir, exist_ok=True)

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Modelo guardado en {MODEL_OUTPUT_PATH}")
print(f"Features esperadas ({len(FEATURE_NAMES)}), en orden:")
for i, nombre in enumerate(FEATURE_NAMES, 1):
    print(f"  {i:2d}. {nombre}")

# Verificación: recargar y predecir 3 filas crudas de prueba
modelo_cargado = joblib.load(MODEL_OUTPUT_PATH)
muestra = X_test.head(3)
pred = modelo_cargado.predict(muestra)
print("Predicciones:", pred.tolist(), "| Reales:", y_test.head(3).tolist())
assert pred.shape[0] == 3, "La verificación de exportación falló"
print("Verificación de exportación OK.")